In [0]:
%sql
-- 4.1 COMMUNICATION METRIC — DAYS_SINCE_LAST_EMAIL
-- Spec: CRM activities where TYPE = Email and MEETING_STATUS = inbox (our silver
-- maps MEETING_STATUS -> status). Exclude automated/access-request messages whose
-- text contains any of the three spec strings (case-insensitive ILIKE):
--   'A new event has been scheduled.'  (deliberately INCLUDED in 4.2 meeting
--   recency — excluded here only)
--   'is requesting access to the following folder'
--   'requests access to an item'
-- Separate from the communication engagement flag (client-message recency drives
-- SLACK_ENGAGED_FLAG; this metric does not). Dedup: one current version per
-- ACTIVITY_ID (DATE_UPDATED then INSERT_DATE), then latest qualifying email per
-- LEAD_ID; recency = DATEDIFF(CURRENT_DATE, DATE(activity_at)).
-- HARDENING (Nov 12 2026, two passes): (1) body_text_quoted is an ARRAY — the
-- original LEFT JOIN + ROW_NUMBER applied the exclusions to ONE arbitrary
-- fragment of the email_body child; an ANY-fragment NOT EXISTS replaced it (no
-- value change, verified on all 6,922 leads). (2) FINAL FORM: exclusions moved
-- to the PARENT body_text_quoted (CAST AS STRING ILIKE — the full text of
-- record). The fragment-extracted child body_text misses the exclusion phrases
-- on 102 of 20,308 matching activities (phrase buried in quoted reply history /
-- a fragment the child write dropped), so the child-based check failed to
-- exclude them — wrong on 8 coaching clients. Parent-sourcing also makes the
-- exclusion predicate IDENTICAL to 3_06's inclusion predicate (the spec's
-- deliberate same-text mirror: excluded here = eligible there). Latest-per-lead
-- rank keeps the deterministic tie-break (id DESC).
CREATE OR REPLACE TABLE crm_ingestion.gold.days_since_last_email AS (
WITH deduped_id_activities_filtered_email AS (
    SELECT
        a.id AS acitvity_id,
        a.lead_id,
        a.activity_at,
        ROW_NUMBER() OVER (PARTITION BY a.id ORDER BY a.date_updated DESC, a.bronze_insert_date DESC) AS id_rn
    FROM
        crm_ingestion.silver.lead_activites_processed a
    WHERE
        a.type = 'Email'
        AND a.status = 'inbox'
        AND NOT (   CAST(a.body_text_quoted AS STRING) ILIKE '%A new event has been scheduled.%'
                 OR CAST(a.body_text_quoted AS STRING) ILIKE '%is requesting access to the following folder%'
                 OR CAST(a.body_text_quoted AS STRING) ILIKE '%requests access to an item%')
),
latest_email_per_lead_id AS (
    SELECT
        *,
        ROW_NUMBER() OVER (PARTITION BY lead_id ORDER BY activity_at DESC, acitvity_id DESC) AS lead_id_rn
    FROM
        deduped_id_activities_filtered_email
    WHERE
        id_rn = 1
)
    SELECT
        lead_id,
        DATEDIFF(CURRENT_DATE, DATE(activity_at)) AS days_since_last_email
    FROM
        latest_email_per_lead_id
    WHERE
        lead_id_rn = 1
)